# CityRain — treino do modelo de câmera fixa no Colab

Para quem não tem Mac M-series nem GPU NVIDIA. Passo a passo completo em `docs/GUIA-TREINO-EQUIPE.md`.

1. **Ambiente de execução → Alterar o tipo de ambiente → GPU (T4)**.
2. Coloque no **seu** Drive, em `MyDrive/cityrain/`, o `cityrain_dados_treino_fixa_<data>.zip` e o `.sha256`
   (no link compartilhado: botão direito → Organizar → Adicionar atalho ao Drive, ou baixar e subir).
3. Rode as células em ordem. Se a sessão cair, rode **todas de novo**: os folds já feitos ficam em
   `MyDrive/cityrain/runs/` e são pulados.

Treino só a partir de **13/10** (congelamento) e depois da revisão visual (`REVISAO.md`).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PASTA = '/content/drive/MyDrive/cityrain'
ZIP = 'cityrain_dados_treino_fixa_20261009.zip'   # <- trocar se o pacote for de outra data
import os; os.makedirs(f'{PASTA}/runs', exist_ok=True); os.makedirs(f'{PASTA}/splits', exist_ok=True); os.makedirs(f'{PASTA}/resultados', exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%cd /content
!test -d CityRain || git clone -q https://github.com/Rodrigomt21/CityRain.git
%cd /content/CityRain
!git pull -q && git log --oneline -1
# torch/torchvision com CUDA já vêm no Colab; o resto vem do requirements
!pip -q install -r ml/requirements-treino.txt

In [ ]:
# dados: confere o checksum e descompacta (só na primeira vez da sessão)
%cd /content/CityRain
!if [ ! -d ml/data/processed/ircnn ]; then (cd "$PASTA" && sha256sum -c "$ZIP.sha256") && unzip -q -n "$PASTA/$ZIP" -d .; fi
# runs, splits e resultados moram no Drive: sobrevivem à queda da sessão
!rm -rf ml/runs ml/data/splits && ln -s "$PASTA/runs" ml/runs && ln -s "$PASTA/splits" ml/data/splits
!mkdir -p ml/resultados && cp -rn ml/resultados/. "$PASTA/resultados/" && rm -rf ml/resultados && ln -s "$PASTA/resultados" ml/resultados
!python ml/scripts/treino/rodar_experimentos_fixa.py checar

Se você colheu dados novos depois do pacote, copie a pasta `ml/data/raw/coleta_fixa` atualizada
para cá antes da próxima célula (os manifests vêm pelo `git pull`).

In [ ]:
# F0 + CV de F1, F2, F3 + escolha (CF5). ~3–4 h na T4. Pode rodar de novo depois de cair.
%cd /content/CityRain
!python ml/scripts/treino/rodar_experimentos_fixa.py tudo

**Pare aqui.** Mande a tabela impressa acima no grupo. Com o ok, rode o modelo final.

In [ ]:
%cd /content/CityRain
!python ml/scripts/treino/rodar_experimentos_fixa.py final
# guarda o ONNX e a config final no Drive para quem for abrir a PR
!cp backend/app/inference/modelos/intensidade_fixa.onnx ml/configs/treino_fixa_final.yaml "$PASTA/"
!test -d backend/app/inference/referencias && cp -r backend/app/inference/referencias "$PASTA/" || true

Depois: baixar `intensidade_fixa.onnx`, `treino_fixa_final.yaml` (e `referencias/`, se F3) e
`resultados/` do Drive para o repo local e abrir a PR (seção 4 do guia).